<a href="https://colab.research.google.com/github/HazemmoAlsady/AWN_Graduation_Project/blob/main/02_batch_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaned Parquet
  #    ↓
# Business Analytics
  #      ↓
# KPIs + Insights + Charts

In [1]:
!pip install pyspark --quiet
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import builtins
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder \
    .appName("RetailAnalytics") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Cleaned Parquet

In [4]:
data = spark.read.parquet(
    "/content/drive/MyDrive/bigdata_project/"
)

print(f"Rows: {data.count():,}")

data.show(5)

Rows: 779,425
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+----------+----+-----+-------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer_ID|       Country|TotalPrice|Year|Month|Quarter|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+----------+----+-----+-------+
| 489460|    84568|GIRLS ALPHABET IR...|     288|2009-12-01 10:46:00| 0.21|    16167.0|United Kingdom|     60.48|2009|   12|      4|
| 489537|    20829|GLITTER HANGING B...|       6|2009-12-01 12:14:00|  2.1|    14040.0|United Kingdom|      12.6|2009|   12|      4|
| 489560|    90087|CRYSTAL SEA HORSE...|      48|2009-12-01 12:56:00| 0.85|    13526.0|United Kingdom|      40.8|2009|   12|      4|
| 489572|    22079|RIBBON REEL HEART...|       5|2009-12-01 13:29:00| 1.65|    17611.0|United Kingdom|      8.25|2009|   12|      4|
| 489573|    21625|VINTAGE UNION JAC...|       7|2009-1

# KPIs

In [5]:
kpis = data.agg(
    count("Invoice").alias("Total_Orders"),

    countDistinct("Invoice")
    .alias("Unique_Invoices"),

    countDistinct("Customer_ID")
    .alias("Unique_Customers"),

    round(sum("TotalPrice"), 2)
    .alias("Total_Revenue"),

    round(avg("TotalPrice"), 2)
    .alias("Avg_Order_Value"),

    round(sum("Quantity"), 0)
    .alias("Total_Items_Sold")
).collect()[0]

NameError: name 'countDistinct' is not defined

In [ ]:
print("="*50)
print("KEY PERFORMANCE INDICATORS")
print("="*50)

print(f"Total Orders      : {kpis['Total_Orders']:,}")
print(f"Unique Invoices   : {kpis['Unique_Invoices']:,}")
print(f"Unique Customers  : {kpis['Unique_Customers']:,}")
print(f"Total Revenue     : £{kpis['Total_Revenue']:,.2f}")
print(f"Avg Order Value   : £{kpis['Avg_Order_Value']:,.2f}")
print(f"Total Items Sold  : {int(kpis['Total_Items_Sold']):,}")

# Product Analysis

In [ ]:
top_products = data.groupBy("Description") \
    .agg(
        round(sum("TotalPrice"), 2)
        .alias("Revenue"),

        sum("Quantity")
        .alias("Units_Sold")
    ) \
    .orderBy(desc("Revenue")) \
    .limit(10)

top_products.show(truncate=False)

# Country Analysis

In [ ]:
top_countries = data.groupBy("Country") \
    .agg(
        round(sum("TotalPrice"), 2)
        .alias("Revenue"),

        countDistinct("Customer_ID")
        .alias("Customers")
    ) \
    .orderBy(desc("Revenue"))

top_countries.show()

# Revenue by Country

In [ ]:
country_pd = top_countries.limit(10).toPandas()

plt.figure(figsize=(10,5))

plt.bar(
    country_pd["Country"],
    country_pd["Revenue"]
)

plt.xticks(rotation=45)

plt.title("Revenue by Country")

plt.ylabel("Revenue (£)")

plt.show()

#

# Top Products Chart

In [ ]:
top_pd = top_products.toPandas()

plt.figure(figsize=(10,6))

plt.barh(
    top_pd["Description"].str[:30][::-1],
    top_pd["Revenue"][::-1]
)

plt.title("Top Products by Revenue")

plt.xlabel("Revenue (£)")

plt.show()

# Sales Trend

In [ ]:
monthly_sales = data.groupBy(
    "Year",
    "Month"
).agg(
    round(sum("TotalPrice"), 2)
    .alias("Revenue")
).orderBy("Year", "Month")

In [ ]:
monthly_pd = monthly_sales.toPandas()

monthly_pd["Date"] = (
    monthly_pd["Year"].astype(str)
    + "-"
    + monthly_pd["Month"].astype(str)
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_pd["Date"],
    monthly_pd["Revenue"],
    marker="o"
)

plt.xticks(rotation=45)

plt.title("Monthly Revenue Trend")

plt.ylabel("Revenue (£)")

plt.show()

# Spark SQL

In [ ]:
data.createOrReplaceTempView(
    "retail_sales"
)
spark.sql("""

SELECT
    Country,
    ROUND(SUM(TotalPrice),2) AS Revenue,
    COUNT(DISTINCT Invoice) AS Orders

FROM retail_sales

GROUP BY Country

ORDER BY Revenue DESC

LIMIT 10

""").show()

# Window Functions

In [ ]:
window_country = Window \
    .partitionBy("Country") \
    .orderBy(desc("Revenue"))

In [ ]:
customer_rev = data.groupBy(
    "Country",
    "Customer_ID"
).agg(
    round(sum("TotalPrice"),2)
    .alias("Revenue")
)

In [ ]:
customer_ranked = customer_rev \
    .withColumn(
        "Rank",
        rank().over(window_country)
    ) \
    .filter(col("Rank") <= 3)

In [ ]:
customer_ranked.show()

#Correlation Analysis

In [ ]:
corr_qty = data.stat.corr(
    "Quantity",
    "TotalPrice"
)

corr_price = data.stat.corr(
    "Price",
    "TotalPrice"
)

print(corr_qty)
print(corr_price)

In [ ]:
top_products.toPandas().to_csv(
    "/content/top_products.csv",
    index=False
)

top_countries.toPandas().to_csv(
    "/content/top_countries.csv",
    index=False
)

monthly_sales.toPandas().to_csv(
    "/content/monthly_sales.csv",
    index=False
)